In [0]:
# Incremental data processing from Bronze to Silver
from pyspark.sql import functions as F
def process_bronze():
    df = (
      spark.readStream.format("cloudFiles")
          .option("cloudFiles.format", "json")
          .schema (schema="key BINARY, value BINARY, topic STRING, partition LONG, offset LONG, timestamp LONG")
          .option("pathGlobFilter", "*.json")
          .load("/Volumes/dev/pro_landing_zone/kafka_sources/books_kafka_row/")
          .withColumn("timestamp", (F.col("timestamp")/1000).cast("timestamp"))
          .withColumn("year_month", F.date_format("timestamp", "yyyy-MM"))
        .writeStream
          .option("checkpointLocation","/Volumes/dev/pro_landing_zone/checkpoints/bronze/")
          .option("mergeSchema", True)
          .partitionBy("topic", "year_month")
          .trigger(availableNow=True)
          .table("dev.multiplex_bronze.kafka_bronze")
    )
    df.awaitTermination()

process_bronze()
  

In [0]:
# Create a function that checks and updates the books_silver layer based on scd type 2
from pyspark.sql import functions as F
from pyspark.sql.window import Window 
def scd_type_2_upsert(microBatchDF, batch):

    
    window = Window.partitionBy("book_id").orderBy(F.col("updated").desc())

    deduped_df = (
        microBatchDF
        .withColumn("rn", F.row_number().over(window))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )


    deduped_df.createOrReplaceTempView("updates")
    sql_query = """
        MERGE INTO dev.silver.books_silver AS target
        USING (
            -- Stage incoming records with a deterministic checksum
            SELECT
                u.book_id as merge_key,
                u.book_id,
                u.title,
                u.author,
                u.price,
                u.updated,
                SHA2(
                    CONCAT_WS(
                        '|',
                        COALESCE(u.title, ''),
                        COALESCE(u.author, ''),
                        COALESCE(CAST(u.price AS STRING), '')
                    ),
                    256
                ) AS checksum
            FROM updates u

            UNION ALL

            SELECT
                NULL as merge_key,
                u.book_id,
                u.title,
                u.author,
                u.price,
                u.updated,
                SHA2(
                    CONCAT_WS(
                        '|',
                        COALESCE(u.title, ''),
                        COALESCE(u.author, ''),
                        COALESCE(CAST(u.price AS STRING), '')
                    ),
                    256
                ) AS checksum
            FROM updates u
            JOIN dev.silver.books_silver bs ON u.book_id = bs.book_id
            WHERE bs.current = true
        ) AS src
        ON target.book_id = merge_key
        AND target.current = TRUE

        -- Expire current record only when data actually changed
        WHEN MATCHED
        AND target.checksum <> src.checksum THEN
            UPDATE SET
                target.current   = FALSE,
                target.end_date  = src.updated
            
        -- Insert new records:
        --  1. brand‑new book_id
        --  2. changed records that were expired by the UPDATE above
        WHEN NOT MATCHED THEN
            INSERT (
                book_id,
                title,
                author,
                price,
                checksum,
                current,
                effective_date,
                end_date
            )
            VALUES (
                src.book_id,
                src.title,
                src.author,
                src.price,
                src.checksum,
                TRUE,
                src.updated,
                NULL
            )
        """
    microBatchDF.sparkSession.sql(sql_query);

In [0]:
"""
        MERGE INTO dev.silver.books_silver AS target 
        USING (            
            WITH staged AS (
                SELECT 
                    u.book_id as merge_key, 
                    u.book_id,
                    u.title,
                    u.author,
                    u.price,
                    u.updated,
                    SHA2(CONCAT_WS('|', COALESCE(u.title, ''), COALESCE(u.author, ''), COALESCE(CAST(u.price AS STRING), '') ), 256) AS checksum
                FROM updates u
            )

            -- Records that should expire the current version
            SELECT
                s.book_id as merge_key,
                s.book_id,
                s.title,
                s.author,
                s.price,
                s.updated,
                s.checksum
            FROM staged s
            JOIN dev.silver.books_silver t
            ON t.book_id = s.book_id AND t.current = true AND t.checksum <> s.checksum
 
            UNION ALL

            SELECT
                NULL AS merge_key,
                s.book_id,
                s.title,
                s.author,
                s.price,
                s.updated,
                s.checksum
            FROM staged s 
        ) staged_updates 
        ON target.book_id = staged_updates.merge_key
        WHEN MATCHED AND target.current = true THEN 
            UPDATE SET target.current = false, target.end_date = staged_updates.updated
        WHEN NOT MATCHED THEN 
            INSERT (
                book_id, 
                title, 
                author, 
                price, 
                current, 
                effective_date, 
                end_date
            ) 
            VALUES (
                staged_updates.book_id, 
                staged_updates.title, 
                staged_updates.author, 
                staged_updates.price, 
                true, 
                staged_updates.updated, 
                NULL
            )        
    """

In [0]:
%sql 
drop table if exists dev.silver.books_silver

In [0]:
%sql 
CREATE OR REPLACE TABLE dev.silver.books_silver(
  book_id STRING, title STRING, author STRING, price DOUBLE, checksum STRING, current BOOLEAN, effective_date TIMESTAMP, end_date TIMESTAMP
)


In [0]:
dbutils.fs.rm("/Volumes/dev/pro_landing_zone/checkpoints/books_silver", True)
dbutils.fs.rm("/Volumes/dev/pro_landing_zone/kafka_sources/books_kafka_row/books_updates_raw/", True)

In [0]:
# Here we model scd type 2 for customer data
from pyspark.sql import functions as F
def process_books():
    book_schema = "book_id STRING, title STRING, author STRING, price DOUBLE, updated TIMESTAMP"

    df_books = (
        spark.readStream.table("dev.multiplex_bronze.kafka_bronze")
            .filter("topic = 'books'")
            .select(
                F.from_json(
                    F.col("value").cast("string"),
                    schema=book_schema

                ).alias('v')
            )
            .select("v.*")
            .writeStream
            .foreachBatch(scd_type_2_upsert)
            .option("checkpointLocation", "/Volumes/dev/pro_landing_zone/checkpoints/books_silver")
            .trigger(availableNow=True)
            .start()    
    )


process_books()

In [0]:
dbutils.fs.cp("s3://dalhussein-courses/DE-Pro/datasets/bookstore/v1/books-updates-streaming", "/Volumes/dev/pro_landing_zone/kafka_sources/books_kafka_row/books_updates_raw/", recurse=True)

In [0]:
process_books()

In [0]:
%sql
select key, cast(value as string) from dev.multiplex_bronze.kafka_bronze where topic = 'books'

In [0]:
%sql
select 
  * 
from dev.silver.books_silver
order by book_id